In [1]:
#%pip install torch
#%pip install gymnasium[other]


In [2]:
# Train SAC Diff Drive

from pathlib import Path
import sys

try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path.cwd()
    if BASE_DIR.name != "Continuous_Diff_Drive":
        BASE_DIR = BASE_DIR / "Continuous_Diff_Drive"

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from SAC_agent import DiffDriveSACAgent
from SAC_env import DiffDriveEnv

BASE_DIR

c:\Users\39324\anaconda3\envs\naml_libraries\lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


WindowsPath('c:/Users/39324/Desktop/NAML_RL_gym/Continuous_Diff_Drive')

In [3]:
## Configuration

OBSTACLES = [
    (2.0, 4.0, 1.5, 0.3),
    (5.0, 2.0, 0.3, 3.0),
    (7.0, 6.0, 1.5, 0.3),
]

ENV_KWARGS = dict(
    room_size       = (10.0, 10.0),
    obstacles       = None,
    random_obst     = True,
    robot_start     = (1.0, 1.0),
    goal_pos        = (8.5, 8.5),

    # SAC beneficia di episodi più lunghi
    max_step        = 1200,

    n_lidar_rays    = 16,
    lidar_max_range = 5.0,
    robot_radius    = 0.3,
    dt              = 0.1,

    render_mode     = "rgb_array",

    obstacle_mode   = "curriculum",
)

# ==========================================================
# TRAINING
# ==========================================================

NUM_EPISODES = 3000

RECORD_EVERY = 500
LOG_EVERY    = 50

# ==========================================================
# SAC HYPERPARAMETERS
# ==========================================================

ACTOR_LR  = 3e-4
CRITIC_LR = 3e-4
ALPHA_LR  = 3e-4

DISCOUNT = 0.99
TAU      = 0.005

BATCH_SIZE = 256
BUFFER_SIZE = 100_000

HIDDEN_DIM = 64

# SAC normalmente usa warmup casuale iniziale
START_STEPS = 5_000

# Aggiornamenti per ogni step ambiente
UPDATES_PER_STEP = 1

# Entropy tuning automatico
AUTO_ENTROPY = True

DEVICE = "cpu"

RUN_TRAINING = True
LOAD_CHECKPOINT_FOR_EVAL = False

In [4]:
## Output Paths

VIDEO_DIR = BASE_DIR / "videos"

TRAINING_VIDEO_DIR = VIDEO_DIR / "training_sac"
EVALUATION_VIDEO_DIR = VIDEO_DIR / "evaluation_sac"

IMAGE_DIR = BASE_DIR / "images"
MODEL_DIR = BASE_DIR / "models"

CHECKPOINT_PATH = MODEL_DIR / "sac_checkpoint.pt"

PLOT_PATH = IMAGE_DIR / "sac_diff_drive_training_curves.png"

TRAINING_NAME_PREFIX = "sac_diff_drive_training"
EVALUATION_NAME_PREFIX = "sac_diff_drive_eval"

for directory in (
    TRAINING_VIDEO_DIR,
    EVALUATION_VIDEO_DIR,
    IMAGE_DIR,
    MODEL_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

VIDEO_DIR

WindowsPath('c:/Users/39324/Desktop/NAML_RL_gym/Continuous_Diff_Drive/videos')

In [5]:
## Environment and Agent

env = DiffDriveEnv(**ENV_KWARGS)

agent = DiffDriveSACAgent(
    env               = env,

    actor_lr          = ACTOR_LR,
    critic_lr         = CRITIC_LR,
    alpha_lr          = ALPHA_LR,

    discount          = DISCOUNT,
    tau               = TAU,

    batch_size        = BATCH_SIZE,
    buffer_size       = BUFFER_SIZE,

    hidden_dim        = HIDDEN_DIM,

    warmup_steps      = START_STEPS,
    #updates_per_step  = UPDATES_PER_STEP,

    automatic_entropy_tuning      = AUTO_ENTROPY,

    device            = DEVICE,
)

agent

In [6]:
## Training

if RUN_TRAINING:

    print("=" * 60)
    print("  DiffDrive - SAC Training")
    print(f"  Episodes           : {NUM_EPISODES}")
    print(f"  Start random steps : {START_STEPS}")
    print(f"  Buffer             : {BUFFER_SIZE}")
    print(f"  Batch size         : {BATCH_SIZE}")
    print(f"  Device             : {DEVICE}")
    print(f"  Videos             : {VIDEO_DIR}")
    print("=" * 60)

    agent.train_recorded(
        num_episodes   = NUM_EPISODES,

        video_folder   = TRAINING_VIDEO_DIR,

        record_every   = RECORD_EVERY,
        log_every      = LOG_EVERY,

        name_prefix    = TRAINING_NAME_PREFIX,

        checkpoint_path = CHECKPOINT_PATH,
        plot_path      = PLOT_PATH,
    )

else:
    print("Training skipped.")

  DiffDrive - SAC Training
  Episodes           : 3000
  Start random steps : 5000
  Buffer             : 100000
  Batch size         : 256
  Device             : cpu
  Videos             : c:\Users\39324\Desktop\NAML_RL_gym\Continuous_Diff_Drive\videos


c:\Users\39324\anaconda3\envs\naml_libraries\lib\site-packages\gymnasium\wrappers\rendering.py:293: UserWarning: WARN: Overwriting existing videos at c:\Users\39324\Desktop\NAML_RL_gym\Continuous_Diff_Drive\videos\training_sac folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(
Training:   1%|▏         | 41/3000 [01:52<2:14:56,  2.74s/it]


KeyboardInterrupt: 

In [ ]:
## Checkpoint Loading

if LOAD_CHECKPOINT_FOR_EVAL:

    import torch

    if not CHECKPOINT_PATH.exists():
        raise FileNotFoundError(
            f"No checkpoint found at {CHECKPOINT_PATH}"
        )

    checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location=DEVICE
    )

    agent.actor.load_state_dict(checkpoint["actor"])

    agent.critic.load_state_dict(checkpoint["critic"])

    agent.critic_target.load_state_dict(
        checkpoint["critic_target"]
    )

    if "log_alpha" in checkpoint:
        agent.log_alpha.data.copy_(checkpoint["log_alpha"])

    print(f"Loaded checkpoint from {CHECKPOINT_PATH}")

else:
    print("Using the current in-memory agent for evaluation.")

In [ ]:
## Evaluation

agent.eval_recorded(
    video_folder = EVALUATION_VIDEO_DIR,
    name_prefix  = EVALUATION_NAME_PREFIX,
    n_episodes   = 3,
)

In [ ]:
## Generated Artifacts

from IPython.display import Image, Video, display

if PLOT_PATH.exists():
    display(Image(filename=str(PLOT_PATH)))

for video_path in sorted(
    TRAINING_VIDEO_DIR.glob(f"{TRAINING_NAME_PREFIX}*.mp4")
):
    print(video_path.name)

for video_path in sorted(
    EVALUATION_VIDEO_DIR.glob(f"{EVALUATION_NAME_PREFIX}*.mp4")
):
    print(video_path.name)
    display(Video(filename=str(video_path), embed=True))